In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import importlib
import test_backward_dp_toc
importlib.reload(test_backward_dp_toc)
from test_backward_dp_toc import perform_backward_dp

V, active_eta, active_alt, active_phase_return = perform_backward_dp()

# Inspection of Results

In [6]:
import networkx as nx
# Load the route graph
G = nx.read_gml("../data/graph/LEMD_EGLL_2023_04_01.gml")
node_to_idx = {node: i for i, node in enumerate(G.nodes())}
idx_to_node = {i: node for i, node in enumerate(G.nodes())}

In [23]:
import torch
import numpy as np

def print_states_at_node(node_id: str, active_eta: torch.Tensor, active_alt: torch.Tensor):
    tensor_eta = active_eta[node_to_idx[node_id]]
    tensor_alt = active_alt[node_to_idx[node_id]]

    # Convert tensors to numpy arrays if they're torch tensors
    arr_eta = tensor_eta.cpu().numpy() if isinstance(tensor_eta, torch.Tensor) else tensor_eta
    arr_alt = tensor_alt.cpu().numpy() if isinstance(tensor_alt, torch.Tensor) else tensor_alt

    # Print table header
    header = f"{'ETA bin':>7} | {'Climb bin Rem':>13} | {'Phase':>5} | {'ETA':>10} | {'Altitude':>10}"
    print(header)
    print('-' * len(header))

    # Iterate over all indices in the array
    for idx in np.ndindex(arr_eta.shape):
        val_eta = arr_eta[idx]
        if not np.isnan(val_eta):
            val_alt = arr_alt[idx] if not np.isnan(arr_alt[idx]) else float('nan')
            eta_bin, climb_bin, phase = idx
            print(f"{eta_bin:7d} | {climb_bin:13d} | {phase:5d} | {val_eta:10.0f} | {val_alt:10.0f}")


In [24]:
print("EGLL")
print_states_at_node("EGLL", active_eta, active_alt)
print("\n"*2)
print("LEMD")
print_states_at_node("LEMD", active_eta, active_alt)

EGLL
ETA bin | Climb bin Rem | Phase |        ETA |   Altitude
---------------------------------------------------------
     60 |             0 |     2 |      43200 |          0



LEMD
ETA bin | Climb bin Rem | Phase |        ETA |   Altitude
---------------------------------------------------------
     39 |             0 |     1 |      37047 |      35000
     39 |             0 |     2 |      37077 |      28056
     39 |            37 |     0 |      36985 |          0
     40 |             0 |     1 |      37282 |      35000
     40 |             0 |     2 |      37439 |      20417
     40 |            37 |     0 |      37405 |          0
     41 |             0 |     1 |      37587 |      35000
     41 |             0 |     2 |      37514 |       7022
     41 |            37 |     0 |      37555 |          0
     42 |             0 |     1 |      37808 |      35000
     42 |             0 |     2 |      37820 |      32113
     42 |            37 |     0 |      37800 |          0


In [26]:
from equinox.helpers.datetimeh import datestr_to_seconds_since_midnight
takeoff_time_ssm = datestr_to_seconds_since_midnight("2023-04-01 10:15:00")
print(f"Takeoff time: {takeoff_time_ssm}")

Takeoff time: 36900.0
